In [ ]:
import json
from pathlib import Path
from typing import Dict, List, Any

# Load models configuration
DATA_PATH = Path(__file__).parent.parent / "data" / "models_config.json"

with open(DATA_PATH, 'r') as f:
    _config = json.load(f)

# Model to provider mapping
MODEL_PROVIDER_MAP: Dict[str, str] = {
    model["model_name"]: model["provider"]
    for model in _config["models"]
}

# Supported models list
SUPPORTED_MODELS: List[str] = list(MODEL_PROVIDER_MAP.keys())

# Provider configurations
PROVIDER_CONFIG: Dict[str, Dict[str, Any]] = _config["providers"]

# Cache settings
DEFAULT_CACHE_TTL: int = _config["cache_settings"]["default_ttl"]
MAX_CACHE_SIZE: int = _config["cache_settings"]["max_cache_size"]

# Provider names
OPENAI_PROVIDER = "openai"
ANTHROPIC_PROVIDER = "anthropic"

# OpenAI models
OPENAI_MODELS = ["gpt-4o", "gpt-4o-mini"]

# Anthropic models
ANTHROPIC_MODELS = ["claude-3-5-sonnet", "claude-3-5-haiku"]



In [ ]:
import time
import hashlib
from typing import Dict, Any, Generator, Optional
from dataclasses import dataclass


@dataclass
class MockResponse:
    """Represents a mock LLM response."""
    content: str
    model: str
    provider: str
    tokens_used: int
    latency_ms: float


class BaseMockProvider:
    """Base class for mock LLM providers."""

    def __init__(self, provider_name: str):
        self.provider_name = provider_name

    def _generate_mock_response(self, prompt: str, model: str) -> str:
        """Generate a deterministic mock response based on input."""
        # Create a hash-based response for determinism
        prompt_hash = hashlib.md5(prompt.encode()).hexdigest()[:8]
        return f"Mock response from {model}: Based on your prompt (hash: {prompt_hash}), here is a helpful response."

    def _calculate_tokens(self, text: str) -> int:
        """Estimate token count (simplified: ~4 chars per token)."""
        return len(text) // 4 + 1

    def complete(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> MockResponse:
        """Generate a complete response."""
        raise NotImplementedError("Subclasses must implement complete()")

    def stream(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> Generator[str, None, None]:
        """Generate a streaming response."""
        raise NotImplementedError("Subclasses must implement stream()")


class OpenAIMockProvider(BaseMockProvider):
    """
    Mock provider simulating OpenAI API behavior.

    Simulates the OpenAI chat completions API format:
    - Uses messages format for input
    - Returns responses in OpenAI-style format
    """

    def __init__(self):
        super().__init__("openai")
        self.supported_models = ["gpt-4o", "gpt-4o-mini"]

    def format_request(self, prompt: str, **kwargs) -> Dict[str, Any]:
        """Format request in OpenAI messages format."""
        return {
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "temperature": kwargs.get("temperature", 0.7),
            "max_tokens": kwargs.get("max_tokens", 1000)
        }

    def complete(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> MockResponse:
        """Generate a complete response in OpenAI style."""
        start_time = time.time()

        # Simulate some processing time
        time.sleep(0.01)  # 10ms simulated latency

        response_text = self._generate_mock_response(prompt, model)

        # Truncate if exceeds max_tokens
        max_chars = max_tokens * 4
        if len(response_text) > max_chars:
            response_text = response_text[:max_chars]

        latency = (time.time() - start_time) * 1000

        return MockResponse(
            content=response_text,
            model=model,
            provider=self.provider_name,
            tokens_used=self._calculate_tokens(response_text),
            latency_ms=latency
        )

    def stream(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> Generator[str, None, None]:
        """Generate a streaming response in OpenAI SSE format."""
        response_text = self._generate_mock_response(prompt, model)

        # Split into words and stream
        words = response_text.split()
        for i, word in enumerate(words):
            # Simulate streaming delay
            time.sleep(0.005)  # 5ms per chunk

            # Format as SSE data
            chunk = word + (" " if i < len(words) - 1 else "")
            yield f"data: {chunk}\n\n"

        # Send done signal
        yield "data: [DONE]\n\n"


class AnthropicMockProvider(BaseMockProvider):
    """
    Mock provider simulating Anthropic API behavior.

    Simulates the Anthropic messages API format:
    - Uses prompt format with Human/Assistant tags
    - Returns responses in Anthropic-style format
    """

    def __init__(self):
        super().__init__("anthropic")
        self.supported_models = ["claude-3-5-sonnet", "claude-3-5-haiku"]

    def format_request(self, prompt: str, **kwargs) -> Dict[str, Any]:
        """Format request in Anthropic prompt format."""
        return {
            "prompt": f"\n\nHuman: {prompt}\n\nAssistant:",
            "temperature": kwargs.get("temperature", 0.7),
            "max_tokens_to_sample": kwargs.get("max_tokens", 1000)
        }

    def complete(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> MockResponse:
        """Generate a complete response in Anthropic style."""
        start_time = time.time()

        # Simulate some processing time
        time.sleep(0.01)  # 10ms simulated latency

        response_text = self._generate_mock_response(prompt, model)

        # Truncate if exceeds max_tokens
        max_chars = max_tokens * 4
        if len(response_text) > max_chars:
            response_text = response_text[:max_chars]

        latency = (time.time() - start_time) * 1000

        return MockResponse(
            content=response_text,
            model=model,
            provider=self.provider_name,
            tokens_used=self._calculate_tokens(response_text),
            latency_ms=latency
        )

    def stream(
        self,
        prompt: str,
        model: str,
        temperature: float = 0.7,
        max_tokens: int = 1000,
        **kwargs
    ) -> Generator[str, None, None]:
        """Generate a streaming response in Anthropic SSE format."""
        response_text = self._generate_mock_response(prompt, model)

        # Split into words and stream
        words = response_text.split()
        for i, word in enumerate(words):
            # Simulate streaming delay
            time.sleep(0.005)  # 5ms per chunk

            # Format as SSE data
            chunk = word + (" " if i < len(words) - 1 else "")
            yield f"data: {chunk}\n\n"

        # Send done signal
        yield "data: [DONE]\n\n"


# Provider registry for easy access
PROVIDERS: Dict[str, BaseMockProvider] = {
    "openai": OpenAIMockProvider(),
    "anthropic": AnthropicMockProvider()
}


def get_provider(provider_name: str) -> Optional[BaseMockProvider]:
    """
    Get a provider instance by name.

    Args:
        provider_name: Name of the provider ('openai' or 'anthropic')

    Returns:
        Provider instance or None if not found
    """
    return PROVIDERS.get(provider_name)



In [ ]:
"""
Model Router implementation for the Multi-LLM Platform API.
COMPLETE THE TODO SECTIONS IN THIS FILE.
"""

from typing import Dict, Any, Generator, Optional, Union
from config import MODEL_PROVIDER_MAP, SUPPORTED_MODELS, OPENAI_PROVIDER, ANTHROPIC_PROVIDER
from mock_providers import (
    BaseMockProvider,
    OpenAIMockProvider,
    AnthropicMockProvider,
    MockResponse,
    PROVIDERS
)


class ModelRouter:
    """
    Routes requests to appropriate LLM providers based on model name.

    This class handles:
    - Determining which provider to use for a given model
    - Formatting requests according to provider specifications
    - Executing requests and returning responses

    Task: Complete the methods below to implement request routing.

    Supported models:
    - gpt-4o, gpt-4o-mini → OpenAI provider
    - claude-3-5-sonnet, claude-3-5-haiku → Anthropic provider
    """

    def __init__(self):
        """Initialize the router with provider instances."""
        self.providers: Dict[str, BaseMockProvider] = PROVIDERS
        self.model_provider_map: Dict[str, str] = MODEL_PROVIDER_MAP

    def get_provider(self, model_name: str) -> BaseMockProvider:
        """
        Get the appropriate provider instance for a model.

        Args:
            model_name: Name of the model (e.g., 'gpt-4', 'claude-3')

        Returns:
            Provider instance that handles this model

        Raises:
            ValueError: If the model is not supported
        """
        if model_name not in self.model_provider_map:
            raise ValueError(f"Model '{model_name}' is not supported. Supported models: {list(self.model_provider_map.keys())}")

        provider_name = self.model_provider_map[model_name]
        return self.providers[provider_name]

    def format_request(
        self,
        provider_name: str,
        prompt: str,
        params: Dict[str, Any]
    ) -> Dict[str, Any]:
        """
        Format a request according to provider specifications.

        Different providers expect different request formats:
        - OpenAI: Uses 'messages' format with role/content
        - Anthropic: Uses 'prompt' format with Human/Assistant tags

        Args:
            provider_name: Name of the provider ('openai' or 'anthropic')
            prompt: The user's prompt text
            params: Additional parameters (temperature, max_tokens, etc.)

        Returns:
            Formatted request dictionary ready for the provider

        Raises:
            ValueError: If the provider is not supported
        """
        providers_name = ['openai', 'anthropic']

        if provider_name not in providers_name:
            raise ValueError(f"Provider '{provider_name}' is not supported. Supported providers: {providers_name}")

        provider = self.providers[provider_name]

        return provider.format_request(prompt, params)

    def route_request(
        self,
        model: str,
        prompt: str,
        stream: bool = False,
        **kwargs
    ) -> Union[MockResponse, Generator[str, None, None]]:
        """
        Route a request to the appropriate provider and return the response.

        This method:
        1. Identifies the correct provider for the model
        2. Routes the request to that provider
        3. Returns either a complete response or a streaming generator

        Args:
            model: Name of the model to use
            prompt: The user's prompt text
            stream: If True, return a streaming generator; if False, return complete response
            **kwargs: Additional parameters (temperature, max_tokens, etc.)

        Returns:
            MockResponse for complete requests, or Generator for streaming

        Raises:
            ValueError: If the model is not supported
        """
        provider = self.get_provider(model)

        if stream:
            return provider.stream(model=model, prompt=prompt, **kwargs)
        else:
            return provider.complete(model=model, prompt=prompt, **kwargs)

    def is_model_supported(self, model_name: str) -> bool:
        """
        Check if a model is supported by this router.

        Args:
            model_name: Name of the model to check

        Returns:
            True if the model is supported, False otherwise
        """
        # Check if model_name is in self.model_provider_map
        return model_name in self.model_provider_map

    def get_provider_name(self, model_name: str) -> Optional[str]:
        """
        Get the provider name for a given model.

        Args:
            model_name: Name of the model

        Returns:
            Provider name or None if model not supported
        """
        # Lookup provider name in self.model_provider_map
        return self.model_provider_map.get(model_name)


In [ ]:
import time
import hashlib
import json
from typing import Dict, Any, Optional
from dataclasses import dataclass
from config import DEFAULT_CACHE_TTL, MAX_CACHE_SIZE


@dataclass
class CacheEntry:
    """Represents a cached response with metadata."""
    response: str
    model: str
    created_at: float
    ttl: int

    @property
    def expires_at(self) -> float:
        """Calculate expiration timestamp."""
        return self.created_at + self.ttl


class ResponseCache:
    """
    TTL-based cache for LLM responses.

    This cache:
    - Stores responses with configurable TTL (time-to-live)
    - Generates deterministic cache keys from request parameters
    - Automatically expires entries after TTL
    - Supports manual invalidation and clearing

    Task: Complete the methods below to implement response caching.
    """

    def __init__(self, default_ttl: int = DEFAULT_CACHE_TTL, max_size: int = MAX_CACHE_SIZE):
        """
        Initialize the cache.

        Args:
            default_ttl: Default time-to-live in seconds for cache entries
            max_size: Maximum number of entries to store
        """
        self.default_ttl = default_ttl
        self.max_size = max_size
        self._cache: Dict[str, CacheEntry] = {}

    def generate_cache_key(
        self,
        model: str,
        prompt: str,
        params: Optional[Dict[str, Any]] = None
    ) -> str:
        """
        Generate a deterministic cache key from request parameters.

        The key should be:
        - Deterministic: Same inputs always produce the same key
        - Unique: Different inputs produce different keys
        - Compact: Use hashing for efficiency

        Args:
            model: Name of the model
            prompt: The user's prompt text
            params: Additional parameters (temperature, max_tokens, etc.)

        Returns:
            A unique cache key string
        """
        request_data = {
            "model": model,
            "prompt": prompt,
            "params": params if params is not None else {}
        }

        request_json = json.dumps(request_data, sort_keys=True)
        return hashlib.sha256(request_json.encode('utf-8')).hexdigest()

    def get(self, key: str) -> Optional[CacheEntry]:
        """
        Retrieve a cached response if it exists and is not expired.

        Args:
            key: Cache key to look up

        Returns:
            CacheEntry if found and not expired, None otherwise
        """
        if key in self._cache:
            entry = self._cache[key]
            if self.is_expired(key):
                # Remove expired entry
                del self._cache[key]
                return None
            return entry

        return None

    def set(
        self,
        key: str,
        response: str,
        model: str,
        ttl: Optional[int] = None
    ) -> None:
        """
        Store a response in the cache.

        Args:
            key: Cache key
            response: Response text to cache
            model: Model that generated the response
            ttl: Time-to-live in seconds (uses default if not specified)
        """
        if not ttl:
            ttl = self.default_ttl

        if len(self._cache) >= self.max_size:
            oldest_key = min(self._cache, key=lambda k: self._cache[k].created_at)
            del self._cache[oldest_key]

        entry = CacheEntry(response=response, model=model, created_at=time.time(), ttl=ttl)
        self._cache[key] = entry

    def is_expired(self, key: str) -> bool:
        """
        Check if a cache entry has expired.

        Args:
            key: Cache key to check

        Returns:
            True if expired or not found, False if valid
        """
        if key not in self._cache:
            return True

        return time.time() >= self._cache[key].expires_at

    def invalidate(self, key: str) -> bool:
        """
        Remove a specific entry from the cache.

        Args:
            key: Cache key to remove

        Returns:
            True if entry was removed, False if not found
        """
        return self._cache.pop(key, None) is not None

    def clear(self) -> int:
        """
        Clear all entries from the cache.

        Returns:
            Number of entries that were cleared
        """
        count = len(self._cache)
        self._cache.clear()
        return count

    def size(self) -> int:
        """
        Get the current number of entries in the cache.

        Returns:
            Number of cached entries
        """
        return len(self._cache)

    def cleanup_expired(self) -> int:
        """
        Remove all expired entries from the cache.

        Returns:
            Number of entries that were removed
        """
        expired_keys = [key for key in self._cache if self.is_expired(key)]
        for key in expired_keys:
            del self._cache[key]
        return len(expired_keys)


In [ ]:
from typing import Optional, List
from fastapi import FastAPI, HTTPException, status
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

from router import ModelRouter
from cache import ResponseCache
from config import SUPPORTED_MODELS, DEFAULT_CACHE_TTL


# Initialize FastAPI app
app = FastAPI(
    title="Multi-LLM Platform API",
    description="Unified API for accessing multiple LLM providers",
    version="1.0.0"
)

# Initialize router and cache
router = ModelRouter()
cache = ResponseCache()


# Request/Response Models
class CompletionRequest(BaseModel):
    """Request model for completion endpoints."""
    model: str = Field(..., description="Model name (e.g., 'gpt-4', 'claude-3')")
    prompt: str = Field(..., min_length=1, max_length=100000, description="Prompt text")
    temperature: Optional[float] = Field(0.7, ge=0.0, le=2.0, description="Sampling temperature")
    max_tokens: Optional[int] = Field(1000, ge=1, le=100000, description="Maximum tokens to generate")
    use_cache: Optional[bool] = Field(True, description="Whether to use caching")


class CompletionResponse(BaseModel):
    """Response model for completion endpoints."""
    response: str = Field(..., description="Generated response text")
    model: str = Field(..., description="Model that generated the response")
    cached: bool = Field(..., description="Whether response was served from cache")
    tokens_used: Optional[int] = Field(None, description="Number of tokens used")


class HealthResponse(BaseModel):
    """Response model for health check."""
    status: str
    supported_models: List[str]
    cache_size: int


class CacheClearResponse(BaseModel):
    """Response model for cache clear operation."""
    cleared: int
    message: str


# API Endpoints

@app.get("/health", response_model=HealthResponse)
async def health_check():
    """
    Health check endpoint.

    Returns the status of the API, supported models, and cache statistics.
    """

    return HealthResponse(
        status="healthy",
        supported_models=SUPPORTED_MODELS,
        cache_size=cache.size()
    )


@app.post("/v1/completions", response_model=CompletionResponse)
async def create_completion(request: CompletionRequest):
    """
    Create a completion using the specified model.

    This endpoint:
    1. Checks if the response is cached (if use_cache is True)
    2. If cached, returns the cached response
    3. If not cached, routes to the appropriate provider
    4. Caches the response (if use_cache is True)
    5. Returns the response with cache status

    Args:
        request: CompletionRequest with model, prompt, and parameters

    Returns:
        CompletionResponse with generated text and metadata

    Raises:
        HTTPException 400: If the model is not supported
        HTTPException 500: If an internal error occurs
    """
    model = request.model

    if not router.is_model_supported(model):
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Model '{model}' is not supported. Supported models: {SUPPORTED_MODELS}"
        )
    if request.use_cache:
        cache_key = cache.generate_cache_key(model=model, prompt=request.prompt, params={
            "temperature": request.temperature,
            "max_tokens": request.max_tokens
        })
        cached_entry = cache.get(cache_key)
        if cached_entry:
            return CompletionResponse(
                response=cached_entry.response.content,
                model=cached_entry.model,
                cached=True,
                tokens_used=cached_entry.response.tokens_used
            )

    try:
        response = router.route_request(
            model=model,
            prompt=request.prompt,
            stream=False,
            temperature=request.temperature,
            max_tokens=request.max_tokens
        )
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Error processing request: {str(e)}"
        )

    if request.use_cache:
        cache.set(key=cache_key, response=response, model=model, ttl=DEFAULT_CACHE_TTL)

    return CompletionResponse(
        response=response.content,
        model=response.model,
        cached=False,
        tokens_used=response.tokens_used
    )


@app.post("/v1/completions/stream")
async def create_streaming_completion(request: CompletionRequest):
    """
    Create a streaming completion using the specified model.

    This endpoint:
    1. Routes to the appropriate provider with stream=True
    2. Returns a StreamingResponse with Server-Sent Events format

    Note: Streaming responses are NOT cached.

    Args:
        request: CompletionRequest with model, prompt, and parameters

    Returns:
        StreamingResponse with SSE-formatted chunks

    Raises:
        HTTPException 400: If the model is not supported
    """
    if not router.is_model_supported(request.model):
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Model '{request.model}' is not supported. Supported models: {SUPPORTED_MODELS}"
        )

    generator = router.route_request(
        model=request.model,
        prompt=request.prompt,
        stream=True,
        temperature=request.temperature,
        max_tokens=request.max_tokens
    )

    return StreamingResponse(generator, media_type="text/event-stream")


@app.delete("/cache", response_model=CacheClearResponse)
async def clear_cache():
    """
    Clear all cached responses.

    Returns:
        Number of entries cleared
    """
    cleared_count = cache.clear()
    return CacheClearResponse(
        cleared=cleared_count,
        message=f"Cleared {cleared_count} cached entries."
    )


@app.get("/models")
async def list_models():
    """
    List all supported models.

    Returns:
        List of supported model names with their providers
    """
    models_info = [
        {"model": model, "provider": router.get_provider_name(model)}
        for model in SUPPORTED_MODELS
    ]
    return models_info
